## Procesamiento de datos con Pandas en el subpaquete `teii.finance`

Ilustramos en este _notebook_ el _pipeline_ principal de procesamiento que implementará posteriormente el paquete `teii.finance`:
1. Lectura de datos del _endpoint_ de Alpha Vantage.
2. Conversión a _dataframe_ convenientemente tipado y con columnas renombradas.
3. Inspección (textual y gráfica) del _dataframe_ obtenido.

### Imports necesarios

In [ ]:
import datetime
import json
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from importlib.resources import files

In [ ]:
pd.__version__

### Constantes

Para conversiones de nombre y tipo de los campos:

In [ ]:
_data_field2name_type = {
            "1. open":                  ("open",     "float"),
            "2. high":                  ("high",     "float"),
            "3. low":                   ("low",      "float"),
            "4. close":                 ("close",    "float"),
            "5. adjusted close":        ("aclose",   "float"),
            "6. volume":                ("volume",   "int"),
            "7. dividend amount":       ("dividend", "float"),
        }

Consulta _ticker_ IBM:

In [ ]:
ticker = 'IBM'
query = f'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY_ADJUSTED&symbol={ticker}&apikey=MY_ALPHA_VANTAGE_API_KEY&data_type=json'

### Realización de la consulta

In [ ]:
download_data = False   # Poner a True si queremos descargarnos los datos de la web

In [ ]:
if download_data:
    # Realizamos la consulta y extraemos los datos de la respuesta:
    response = requests.get(query)
    json_data_downloaded = response.json()
    with open(f"TIME_SERIES_WEEKLY_ADJUSTED.IBM.json", 'w') as outfile:
        outfile.write(json.dumps(json_data_downloaded, indent=4))
else: 
    # Abrimos el fichero JSON con datos descargados con anterioridad:
    json_file = files('teii.finance.data').joinpath('TIME_SERIES_WEEKLY_ADJUSTED.IBM.json')
    with json_file.open('r') as json_fid:
        json_data_downloaded = json.load(json_fid)
print(json.dumps(json_data_downloaded, indent=2))

### Inspección del JSON

In [ ]:
json_metadata = json_data_downloaded['Meta Data']
print(json.dumps(json_metadata, indent=2))

In [ ]:
json_data = json_data_downloaded['Weekly Adjusted Time Series']
print(json.dumps(json_data, indent=2))

### Conversión a _dataframe_ correctamente tipado

Construimos el _dataframe_ a partir del diccionario de diccionarios (`json_data`) con los datos de las series temporales (que estarán ahora disponibles como columnas del _dataframe_):

In [ ]:
# Usamos parámetro orient='index' del método from_dictpara que las claves del dict (cadenas de fechas) queden como índice:
data_frame = pd.DataFrame.from_dict(json_data, orient='index')

# Tipos de datos del índice y de las columnas (tal como se han leído originalmente del json):
print("Index data type:", data_frame.index.dtype)
print("Column data types:")
print(data_frame.dtypes)

Diccionarios para las conversiones de nombre (primero) y de tipo (después) de las columnas:

In [ ]:
dict_name_conversions = {key: name_type[0] for key, name_type in _data_field2name_type.items()}
dict_name_conversions

In [ ]:
dict_type_conversions = {name_type[0]: name_type[1] for key, name_type in _data_field2name_type.items()}
dict_type_conversions

Empleamos los diccionarios anteriores para la conversión de tipos y renombrado de columnas del _dataframe_:

In [ ]:
# Renombramos las columnas con nombres más cortos y sin espacios, y también con nombres más significativos 
# (open, high, low, close, aclose, volume, dividend):
data_frame = data_frame.rename(columns=dict_name_conversions)

# Convertimos los tipos de datos de las columnas a los tipos adecuados (float para los campos numéricos y 
# datetime para el índice de fechas):
data_frame = data_frame.astype(dtype=dict_type_conversions)

# Y ahora convertimos el índice de fechas a tipo datetime:
data_frame.index = data_frame.index.astype("datetime64[ns]")

# Tipos de datos del índice y de las columnas (tras la conversión)
print("\nIndex data type:", data_frame.index.dtype)
print("Column data types:")
print(data_frame.dtypes)

Ahora ya podemos inspeccionar mejor el _dataframe_:

In [ ]:
data_frame

In [ ]:
data_frame.describe()

Finalmente, ordenamos del _dataframe_ en sentido ascendente de fechas:

In [ ]:
data_frame = data_frame.sort_index(ascending=True)
data_frame

A partir de aquí ya tenemos en `data_frame` los datos con los que vamos a trabajar, así que podemos usar este notebook para hacer las pruebas necesarias para extraer lo que nos pedirán en los métodos de `TimesSeriesFinanceClient()`.

## Gráfico de la serie de datos de cierre semanal

Extraemos la serie temporal de precios de cierre (close) como un objeto pandas Series, y lo mostramos gráficamente (al estilo de `example.py`):

In [ ]:
data_frame['close']

In [ ]:
pandas_series = data_frame['close']
pandas_series.plot(xlabel='Fecha', ylabel='Precio en USD', title=f"Evolución del precio de {ticker}");
